In [1]:
import os

import matplotlib.pyplot as plt
import pandas as pd

%run analysis_utils.py
%run fig2_plot_utils.py

RESULTS_PATH = os.path.expanduser(
    '~/scFM_eval/results/embedding_bootstrap/embedding.metrics.bootstrap.csv'
)
print(RESULTS_PATH)
GROUP_ORDER = ["Baseline", "Geneformer", "Other", "scGPT"]
PLOT_DIR = "./plots"

results_10_runs = pd.read_csv(RESULTS_PATH)
results_10_runs["method"] = results_10_runs["model"]

exclude = (
    results_10_runs.model_display.str.contains("continue")
    | results_10_runs.model_display.str.contains("Full")
    | results_10_runs.model_display.isin(["PCA [100]", "PCA [50]"])
)
results_10_runs = results_10_runs.loc[~exclude].copy()


/home/haitham/scFM_eval/results/embedding_bootstrap/embedding.metrics.bootstrap.csv


In [2]:
results_10_runs

,run,model,model_display,group,leiden_resolution,n_leiden_clusters,NMI_cluster/label,ARI_cluster/label,ASW_label,graph_conn,...,iLISI,cLISI,kBET,avg_bio,knn_label_purity,knn_batch_purity,batch_pred_acc,kNN_label_acc,kNN_label_f1_macro,method
0,0,state_se600m_epoch16,STATE,Other,0.1,9,0.699710,0.622641,0.520511,0.982418,...,0.035291,1.0,0.523249,0.614288,0.925300,0.356993,0.4825,0.9655,0.950979,state_se600m_epoch16
1,1,state_se600m_epoch16,STATE,Other,0.2,11,0.685022,0.604798,0.520183,0.966231,...,0.034006,1.0,0.402799,0.603335,0.919407,0.359300,0.4840,0.9610,0.959287,state_se600m_epoch16
2,2,state_se600m_epoch16,STATE,Other,0.6,19,0.682021,0.607740,0.521345,0.972359,...,0.033649,1.0,0.393846,0.603702,0.916293,0.359860,0.4735,0.9655,0.958372,state_se600m_epoch16
3,3,state_se600m_epoch16,STATE,Other,0.3,15,0.669311,0.589274,0.520340,0.976387,...,0.034808,1.0,0.409206,0.592975,0.908807,0.358347,0.4850,0.9565,0.948539,state_se600m_epoch16
4,4,state_se600m_epoch16,STATE,Other,0.4,15,0.680434,0.594392,0.521387,0.976525,...,0.035115,1.0,0.391587,0.598738,0.915913,0.358820,0.4860,0.9645,0.950921,state_se600m_epoch16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
185,5,cellplm_85M-20231027,CellPLM,Other,0.1,7,0.855508,0.889043,0.593640,0.979457,...,0.028240,1.0,0.342252,0.779397,0.948740,0.374813,0.4635,0.9560,0.951691,cellplm_85M-20231027
186,6,cellplm_85M-20231027,CellPLM,Other,0.1,6,0.823974,0.865157,0.593392,0.974559,...,0.028730,1.0,0.330549,0.760841,0.944287,0.375127,0.4755,0.9585,0.933014,cellplm_85M-20231027
187,7,cellplm_85M-20231027,CellPLM,Other,0.1,7,0.859281,0.896307,0.593546,0.979042,...,0.028271,1.0,0.317957,0.783045,0.948960,0.378167,0.4900,0.9565,0.954588,cellplm_85M-20231027
188,8,cellplm_85M-20231027,CellPLM,Other,0.1,7,0.828142,0.872493,0.592894,0.979642,...,0.028372,1.0,0.326841,0.764510,0.948200,0.374880,0.4650,0.9560,0.963174,cellplm_85M-20231027


In [3]:
METRIC_DISPLAY_NAMES = {
    "NMI_cluster/label": "Cluster–Label NMI",
    "ARI_cluster/label": "Cluster–Label ARI",
    "ASW_label": "Cell-Type Separation",
    "graph_conn": "Graph Connectivity",
    "ASW_batch": "Batch Mixing ASW",
    "ASW_label/batch": "Biology–Batch ASW Ratio",
    "batch_ASW": "Batch Mixing Score",
    "PCR_batch": "Batch Effect PCR",
    "iLISI": "Batch Diversity",
    "cLISI": "Cell-Type Purity",
    "kBET": "Batch Mixing",
    "avg_bio": "Overall Biology Score",
    "knn_label_purity": "Neighbor Label Purity",
    "knn_batch_purity": "Neighbor Batch Purity",
    "batch_pred_acc": "Batch Predictability",
    "kNN_label_acc": "kNN Label Accuracy",
    "kNN_label_f1_macro": "kNN Label F1",
}

results_10_runs_renamed = results_10_runs.rename(columns=METRIC_DISPLAY_NAMES)
metrics = list(METRIC_DISPLAY_NAMES.values())


In [4]:
results_10_runs_renamed.model_display.value_counts()

model_display
STATE             10
scVI              10
SCimiarity        10
scGPT             10
scGPT [cancer]    10
scFoundation      10
scConcept         10
PCA [20]          10
Nicheformer       10
HVG               10
GF-V2-Deep        10
GF-V2 [cancer]    10
GF-V2             10
GF-V1             10
CellPLM           10
Name: count, dtype: int64

In [5]:
results_10_runs_renamed.model_display.value_counts()

model_display
STATE             10
scVI              10
SCimiarity        10
scGPT             10
scGPT [cancer]    10
scFoundation      10
scConcept         10
PCA [20]          10
Nicheformer       10
HVG               10
GF-V2-Deep        10
GF-V2 [cancer]    10
GF-V2             10
GF-V1             10
CellPLM           10
Name: count, dtype: int64

In [6]:
def prep_plot_df(df, source_col, plot_col):
    out = df[["model_display", source_col, "group"]].copy()
    out.columns = ["model", plot_col, "group"]
    return out


def plot_metric(df, metric_col, ylim=None, save_path=None):
    kwargs = {
        "group_order": GROUP_ORDER,
        "point_size": 2.8,
        "point_alpha": 0.55,
    }
    if ylim is not None:
        kwargs["ylim"] = ylim
    if save_path is not None:
        kwargs["save_path"] = save_path
    return plot_metric_grouped_single_axis(df, metric_col=metric_col, **kwargs)


In [7]:
os.makedirs(PLOT_DIR, exist_ok=True)


In [8]:
for metric_name in metrics:
    metric_df = prep_plot_df(results_10_runs_renamed, metric_name, metric_name)
    fig, ax = plot_metric(
        metric_df,
        metric_col=metric_name,
        save_path=os.path.join(PLOT_DIR, f"{metric_name}.png"),
    )
    plt.close(fig)
